# K-Means Clustering Plots Testing Notebook

This notebook is for manual testing of K-means clustering and quartile distribution plots.
It uses functions from `postprocess_functions.py` and `plot_kmeans_functions.py`.

K-means clustering groups the input parameter space and shows how target quartiles are distributed across clusters.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add parent directory to path for imports
sys.path.insert(0, str(Path().resolve().parent))

from ddstartup.postprocessing.postprocess_functions import (
    load_h5_to_dataframe,
    get_input_parameters,
    scale_target,
    apply_filters,
    find_latest_h5_file
)
from ddstartup.postprocessing.plot_kmeans_functions import (
    cluster_and_quartile_bar
)
from ddstartup.utils.tools import PARAM_UNITS
from sklearn.preprocessing import StandardScaler

print("✅ Imports successful")

## Configuration

In [ ]:
# Configuration - specify directory or files

# Option 1: Automatic - find latest file in specified directory (DEFAULT)
outputs_dir = Path('../outputs')
files_to_analyze = []

latest_h5_file = find_latest_h5_file(outputs_dir)
if latest_h5_file:
    print(f"📂 Auto-detected folder: {latest_h5_file.parent.name}")
    print(f"📄 Latest file: {latest_h5_file.name}")
    files_to_analyze = [latest_h5_file]

# Option 2: Analyze all files in a specific folder
# outputs_dir = Path('../outputs/20251008_081422_parametric_T_seeded')
# files_to_analyze = sorted(outputs_dir.glob('*.h5'))
# print(f"📂 Using folder: {outputs_dir.name}")
# print(f"📄 Found {len(files_to_analyze)} file(s): {[f.name for f in files_to_analyze]}")

# Target variable to analyze
target = 'Q_fusion'  # Change this to your desired target variable
print(f"\n🎯 Target variable: {target}")

## Load and Prepare Data

In [ ]:
# Load data
if not files_to_analyze:
    raise ValueError("No files to analyze. Please check the configuration.")

df = load_h5_to_dataframe(files_to_analyze)
print(f"📊 Loaded dataframe with shape: {df.shape}")
print(f"📋 Columns: {list(df.columns)}")

# Get input parameters
inputs = get_input_parameters(df)
print(f"\n🔧 Input parameters ({len(inputs)}): {inputs}")

# Check if target exists
if target not in df.columns:
    print(f"❌ Target '{target}' not found in dataframe")
    print(f"Available columns: {list(df.columns)}")
    raise ValueError(f"Target '{target}' not in dataframe")

print(f"\n📈 Target '{target}' range: [{df[target].min():.2e}, {df[target].max():.2e}]")
print(f"📊 Target statistics:")
print(df[target].describe())

## Apply Filters (Optional)

In [ ]:
# Optional: Apply filters
filters = {
    # Example: 'Q_fusion': {'min': 0},  # Only positive Q_fusion
    # Example: 'Ti_0': {'min': 5e3, 'max': 20e3},  # Temperature range
}

if filters:
    df_filtered = apply_filters(df, filters)
    print(f"🔍 Applied filters: {filters}")
    print(f"📊 Filtered dataframe shape: {df_filtered.shape} (was {df.shape})")
    df = df_filtered
else:
    print("No filters applied")

## Test 1: Basic K-Means Clustering (5 clusters)

In [ ]:
# Create basic k-means clustering plot
if len(inputs) > 0:
    # Create output directory for test plots
    test_outputs_dir = Path('../outputs/manual_test_kmeans')
    test_outputs_dir.mkdir(parents=True, exist_ok=True)
    
    n_clusters = 5
    print(f"Creating K-means clustering plot with {n_clusters} clusters...\n")
    
    kmeans, crosstab = cluster_and_quartile_bar(
        df,
        inputs=inputs,
        target=target,
        outputs_dir=test_outputs_dir,
        n_clusters=n_clusters,
        plot_name=f'kmeans_{target}_k{n_clusters}',
        save_csv=True
    )
    
    print(f"✅ Plot saved to {test_outputs_dir}")
    print(f"   - PNG: kmeans_{target}_k{n_clusters}.png")
    print(f"   - CSV: kmeans_{target}_k{n_clusters}_cluster_centers.csv")
    
    # Display the crosstab
    print(f"\nQuartile distribution per cluster (normalized):")
    print("="*60)
    print(crosstab)
    print("="*60)
    
    # Display the saved image
    from IPython.display import Image, display
    display(Image(filename=test_outputs_dir / f'kmeans_{target}_k{n_clusters}.png'))
else:
    print("❌ No input parameters available for clustering")

## Test 2: Analyze Cluster Centers

In [ ]:
# Examine the cluster centers
if len(inputs) > 0:
    # Load the saved cluster centers CSV
    centers_csv = test_outputs_dir / f'kmeans_{target}_k{n_clusters}_cluster_centers.csv'
    
    if centers_csv.exists():
        centers_df = pd.read_csv(centers_csv)
        
        print("Cluster Centers:")
        print("="*80)
        print(centers_df)
        print("="*80)
        
        # Show which parameters vary most between clusters
        print("\nParameter variation across clusters (std dev):")
        for param in inputs:
            if param in centers_df.columns:
                std = centers_df[param].std()
                print(f"  {param:20s} : {std:.3e}")
    else:
        print("⚠️ Cluster centers CSV not found")

## Test 3: Different Number of Clusters

In [ ]:
# Test with different numbers of clusters
if len(inputs) > 0:
    cluster_counts = [3, 7, 10]
    
    print("Testing different numbers of clusters...\n")
    
    for k in cluster_counts:
        print(f"\nCreating clustering with k={k}...")
        
        kmeans_k, crosstab_k = cluster_and_quartile_bar(
            df,
            inputs=inputs,
            target=target,
            outputs_dir=test_outputs_dir,
            n_clusters=k,
            plot_name=f'kmeans_{target}_k{k}',
            save_csv=False  # Don't save CSV for this test
        )
        
        print(f"  Inertia (sum of squared distances): {kmeans_k.inertia_:.2e}")
        print(f"  Number of iterations: {kmeans_k.n_iter_}")
    
    print(f"\n✅ All cluster variations saved to {test_outputs_dir}")

## Test 4: Elbow Method for Optimal K

In [ ]:
# Use elbow method to find optimal number of clusters
if len(inputs) > 0:
    from sklearn.cluster import KMeans
    
    print("Performing elbow method analysis...\n")
    
    # Prepare data
    X = df[inputs].fillna(0)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Test different k values
    k_range = range(2, 11)
    inertias = []
    
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=0)
        km.fit(X_scaled)
        inertias.append(km.inertia_)
        print(f"  k={k:2d} : inertia = {km.inertia_:.3e}")
    
    # Plot elbow curve
    plt.figure(figsize=(10, 6))
    plt.plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
    plt.xlabel('Number of Clusters (k)', fontsize=12)
    plt.ylabel('Inertia (Sum of Squared Distances)', fontsize=12)
    plt.title(f'Elbow Method for Optimal k - {target}', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(test_outputs_dir / f'elbow_method_{target}.png', dpi=150)
    plt.show()
    
    print(f"\n✅ Elbow plot saved to {test_outputs_dir}/elbow_method_{target}.png")
    print("\n💡 Look for the 'elbow' point where inertia decrease slows down")

## Test 5: Cluster Analysis with Target Statistics

In [ ]:
# Analyze target statistics for each cluster
if len(inputs) > 0 and 'cluster' in df.columns:
    print(f"Target statistics per cluster for '{target}':")
    print("="*80)
    
    cluster_stats = df.groupby('cluster')[target].agg([
        'count', 'mean', 'std', 'min', 'max'
    ])
    
    print(cluster_stats)
    print("="*80)
    
    # Find best and worst clusters
    best_cluster = cluster_stats['mean'].idxmax()
    worst_cluster = cluster_stats['mean'].idxmin()
    
    print(f"\nBest cluster (highest mean {target}): Cluster {best_cluster}")
    print(f"  Mean {target}: {cluster_stats.loc[best_cluster, 'mean']:.3e}")
    print(f"  Count: {cluster_stats.loc[best_cluster, 'count']:.0f} samples")
    
    print(f"\nWorst cluster (lowest mean {target}): Cluster {worst_cluster}")
    print(f"  Mean {target}: {cluster_stats.loc[worst_cluster, 'mean']:.3e}")
    print(f"  Count: {cluster_stats.loc[worst_cluster, 'count']:.0f} samples")
    
    # Plot box plots for each cluster
    plt.figure(figsize=(10, 6))
    df.boxplot(column=target, by='cluster', ax=plt.gca())
    plt.ylabel(target)
    plt.xlabel('Cluster')
    plt.title(f'{target} Distribution by Cluster')
    plt.suptitle('')  # Remove default title
    plt.tight_layout()
    plt.savefig(test_outputs_dir / f'cluster_boxplot_{target}.png', dpi=150)
    plt.show()
    
    print(f"\n✅ Cluster boxplot saved")

## Test 6: Cluster Visualization in 2D (PCA)

In [ ]:
# Visualize clusters in 2D using PCA
if len(inputs) > 1 and 'cluster' in df.columns:
    from sklearn.decomposition import PCA
    
    print("Visualizing clusters in 2D using PCA...\n")
    
    # Prepare data
    X = df[inputs].fillna(0)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Apply PCA
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    
    print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
    print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.2%}")
    
    # Plot
    plt.figure(figsize=(12, 8))
    scatter = plt.scatter(
        X_pca[:, 0], X_pca[:, 1],
        c=df['cluster'], cmap='tab10',
        alpha=0.6, s=30
    )
    plt.colorbar(scatter, label='Cluster')
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
    plt.title('Cluster Visualization in 2D (PCA)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(test_outputs_dir / f'cluster_pca_2d_{target}.png', dpi=150)
    plt.show()
    
    print(f"\n✅ PCA visualization saved")

## Test 7: Multiple Targets Comparison

In [ ]:
# Compare clustering results for multiple targets
output_columns = [col for col in df.columns if col not in inputs and col not in ['index', 'cluster', 'quartile']]
targets_to_test = output_columns[:3] if len(output_columns) >= 3 else output_columns

if len(targets_to_test) > 1:
    print(f"Creating clustering plots for multiple targets: {targets_to_test}\n")
    
    for tgt in targets_to_test:
        print(f"\n{'='*60}")
        print(f"Target: {tgt}")
        print(f"{'='*60}")
        
        kmeans_tgt, crosstab_tgt = cluster_and_quartile_bar(
            df,
            inputs=inputs,
            target=tgt,
            outputs_dir=test_outputs_dir,
            n_clusters=5,
            plot_name=f'kmeans_{tgt}',
            save_csv=False
        )
        
        # Show which cluster has the most Q4 samples
        if 'Q4' in crosstab_tgt.columns:
            best_cluster = crosstab_tgt['Q4'].idxmax()
            q4_proportion = crosstab_tgt.loc[best_cluster, 'Q4']
            print(f"  Cluster {best_cluster} has highest Q4 proportion: {q4_proportion:.1%}")
    
    print(f"\n✅ All clustering plots saved to {test_outputs_dir}")
else:
    print("ℹ️ Not enough output variables for multi-target test")

## Summary

### K-Means Clustering Functions Tested:

1. ✅ **cluster_and_quartile_bar**
   - Clusters input parameter space using K-means
   - Divides target into quartiles (Q1-Q4)
   - Creates stacked bar chart showing quartile distribution per cluster
   - Saves cluster centers to CSV

### Key Concepts:

**K-Means Clustering:**
- Groups similar parameter combinations
- Uses standardized inputs for fair comparison
- Number of clusters (k) is user-specified

**Quartile Distribution:**
- Q1: Lowest 25% of target values
- Q2: 25-50% of target values
- Q3: 50-75% of target values
- Q4: Highest 25% of target values

**Interpretation:**
- Each cluster represents a distinct region in parameter space
- Bar height shows proportion of samples in each quartile
- Clusters with high Q4 proportion → good parameter combinations
- Clusters with high Q1 proportion → poor parameter combinations

### Use Cases:

1. **Identify optimal parameter regions**
   - Look for clusters with high Q4 proportion
   - Extract cluster centers from CSV

2. **Understand parameter interactions**
   - Compare cluster centers
   - See which parameters vary most between clusters

3. **Guide optimization**
   - Focus on parameter ranges in best clusters
   - Avoid parameter ranges in worst clusters

### Choosing Number of Clusters:

**Methods:**
- Elbow method: Look for "elbow" in inertia plot
- Domain knowledge: Choose k based on problem context
- Silhouette score: Measure cluster quality

**General Guidelines:**
- Too few clusters: Lose detail, oversimplify
- Too many clusters: Overfit, hard to interpret
- Typical range: 3-10 clusters

### Output Files:

1. **PNG**: Stacked bar chart of quartile distributions
2. **CSV**: Cluster centers (original parameter scales)
   - Use these values for optimization or further analysis
   - Each row = one cluster center
   - Each column = one input parameter

### Additional Analyses:

- **PCA visualization**: See clusters in 2D space
- **Box plots**: Compare target distributions per cluster
- **Statistics**: Mean, std, min, max of target per cluster
- **Multi-target**: Compare clustering across different outputs